# 01. Оценка качества исходного baseline

Проверяется **существующий** артефакт `artifacts/baseline_model.joblib` на исходном фиксированном test-срезе. Входной датасет должен быть **тем же файлом**, что использовался при обучении, иначе проверка остановится по SHA-256. Сам датасет не включён в архив проекта. Метки: `NORMAL → neutral`, `INSULT/OBSCENITY → offensive`, `THREAT → toxic`.

Запуск из корня репозитория: `DATA_PATH=data/dataset.txt.zip jupyter lab notebooks/01_quality_evaluation.ipynb`.
Этот ноутбук не обучает модель, не меняет её и не подбирает гиперпараметры по тесту.

In [ ]:
import os, sys, json, hashlib
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if not (ROOT / 'mlops').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from mlops.data import CLASSES, read_labeled_dataset, split_like_original, dataset_sha256
DATA_PATH = Path(os.getenv('DATA_PATH', str(ROOT / 'data/dataset.txt.zip')))
MODEL_PATH = Path(os.getenv('MODEL_PATH', str(ROOT / 'artifacts/baseline_model.joblib')))
METADATA_PATH = Path(os.getenv('MODEL_METADATA_PATH', str(ROOT / 'artifacts/baseline_metadata.json')))
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Нет исходного датасета: {DATA_PATH}. Поместите dataset.txt.zip в папку data/.')
metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8'))
expected = metadata['dataset_sha256']
actual = dataset_sha256(DATA_PATH)
assert expected == actual, 'SHA-256 датасета не совпадает с обучением baseline; тест недостоверен'
assert int(metadata['seed']) == 42, 'Тестовый split в исходном notebook использует seed=42'
model = joblib.load(MODEL_PATH)  # Только доверенные локальные артефакты
assert set(model.classes_) == set(CLASSES)
print('SHA-256 совпадает; модель:', metadata['model'])

## Воспроизведение отложенного теста без пересечений

In [ ]:
raw, malformed = read_labeled_dataset(DATA_PATH)
train, valid, test = split_like_original(raw, seed=int(metadata['seed']))
assert len(test) == metadata['test_rows'], 'Размер test отличается от исходного'
assert len(valid) == metadata['val_rows'], 'Размер validation отличается от исходного'
print('Некорректных строк:', malformed)
print('Размеры:', {'train':len(train),'validation':len(valid),'test':len(test)})
display(pd.DataFrame({'train':train.label.value_counts(), 'validation':valid.label.value_counts(), 'test':test.label.value_counts()}).reindex(CLASSES).fillna(0).astype(int))

## Accuracy, macro-F1, classification report и recall класса toxic

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
truth = test.label.to_numpy()
pred = model.predict(test.text_clean)
metrics = {'accuracy':float(accuracy_score(truth,pred)),
           'macro_f1':float(f1_score(truth,pred,labels=CLASSES,average='macro',zero_division=0)),
           'weighted_f1':float(f1_score(truth,pred,labels=CLASSES,average='weighted',zero_division=0))}
report = classification_report(truth,pred,labels=CLASSES,output_dict=True,zero_division=0)
display(pd.DataFrame(report).T.round(4))
print('Metrics:',json.dumps(metrics,ensure_ascii=False,indent=2))
print('Stored macro-F1:',metadata['test_macro_f1'],'recomputed:',metrics['macro_f1'])
print('Stored toxic recall:',metadata['test_toxic_recall'],'recomputed:',report['toxic']['recall'])

## Матрица ошибок и уверенность

In [ ]:
ConfusionMatrixDisplay.from_predictions(truth,pred,labels=CLASSES,xticks_rotation=30,cmap='Blues',values_format='d')
plt.tight_layout()
plt.show()
if hasattr(model,'predict_proba'):
    probs=model.predict_proba(test.text_clean)
    assert np.all(np.isfinite(probs)) and np.allclose(probs.sum(axis=1),1,atol=1e-6)
    summary=pd.DataFrame({'predicted':pred,'correct':truth==pred,'confidence':probs.max(axis=1)})
    display(summary.groupby('correct').confidence.agg(['mean','median','count']).round(3))

## Выводы

Оценка справедлива только при совпадении SHA исходного датасета и сохранённого baseline. Тестовая выборка не используется для выбора модели или настройки порога. Отчёты содержат агрегаты, **без текстов комментариев**. Для достоверного сравнения с сохранёнными метриками также требуется совместимая версия `scikit-learn` (для исходного артефакта 1.9.1).